In [64]:
#load data zwn
import pandas as pd
raw_zwn = pd.read_csv("../data/raw/zwn_meldungen.csv")
#Clean data
#select useful columns
processed_zwn = raw_zwn[["service_name","requested_datetime","e","n"]]

#define new columns names
new_names = {
"service_name":"category",
"e":"East",
"n":"North",
"requested_datetime":"report_time",
}
processed_zwn= processed_zwn.rename(columns=new_names)

# convert datatype of "report_time" to datetime64
processed_zwn["report_time"] = pd.to_datetime(processed_zwn["report_time"], format ="%Y-%m-%dT%H:%M:%S")

# create geometry category and geodataframe
import geopandas as gpd
from shapely.geometry import Point
processed_zwn = gpd.GeoDataFrame(processed_zwn,geometry=gpd.points_from_xy(processed_zwn["East"], processed_zwn["North"])
)
# select useful columns
processed_zwn= processed_zwn[["category","report_time","geometry"]]
processed_zwn.head((3))

# check missing values
#missing_count = processed_zwn[["category","report_time","geometry"]].isna().sum()
#print(f"The table shows the missing values in the dataframe {missing_count}")

#define missing CRS (CH1903+ / LV95)
processed_zwn = processed_zwn.set_crs(epsg=2056)

# load data quartiere
raw_quartiere = pd.read_csv("../data/raw/quartiere_zürich.csv")
processed_quartiere = raw_quartiere[["qname","geometry"]]
processed_quartiere.head(3)

#define new columns names
new_names1 = {
"qname":"Quartier",
    "geometry": "Geometry"
}
processed_quartiere= processed_quartiere.rename(columns=new_names1)

# check missing values
#missing_count = processed_quartiere[["Quartier","Geometry"]].isna().sum()
#print(f"The table shows the missing values in the dataframe {missing_count}")

#transform geometry datatype
#wkt.loads transform string datatype to geometry data type
from shapely import wkt
processed_quartiere["Geometry"] = processed_quartiere["Geometry"].apply(wkt.loads)

# create geodataframe to interprate "Geometry" as a geometry column
processed_quartiere = gpd.GeoDataFrame(
    processed_quartiere,
    geometry="Geometry")

#define missing CRS (CH1903+ / LV95)
processed_quartiere = processed_quartiere.set_crs(epsg=2056)

#download processed csv-files
#processed_zwn.to_file("../data/processed/processed_zwn.gpkg", driver="GPKG")
#processed_quartiere.to_file("../data/processed/processed_quartiere.gpkg", driver="GPKG")

# Question 1: "Welche Problemkategorien werden im jeweiligen statistischen Qaurtier am häufigsten gemeldet?

#perform spatial join
zwn_with_quartiere = gpd.sjoin(processed_zwn,processed_quartiere, how="left", predicate ="within")


# Check NaN values in spatial join: 8 NaN values out of 72'411 reports
#zwn_with_quartiere["Quartier"].isna().sum()
#zwn_with_quartiere[zwn_with_quartiere["Quartier"].isna()]
#zwn_with_quartiere["Quartier"].notna().sum()

#count how many times a specific category in a specific quartier is reported
#reset_index wird verwendet, da Output sind Index(also Quartiernamen ist der Index) ist und man will wieder normale Spalten
#name="count" ist der Name der Spalte, sonst wäre er 0

count = zwn_with_quartiere.groupby(["Quartier","category"]).size().reset_index(name="count")
count.head(10)

#idx = count.groupby("Quartier")["count"].max()

#print(idx)
#result = count.loc[idx]
#print(result)

#print(count)







,Quartier,category,0
0,Affoltern,Abfall/Sammelstelle,775
1,Affoltern,Allgemein,109
2,Affoltern,Beleuchtung/Uhren,245
3,Affoltern,Brunnen/Hydranten,30
4,Affoltern,Graffiti,133
5,Affoltern,Grünflächen/Spielplätze,308
6,Affoltern,Schädlinge,40
7,Affoltern,Signalisation/Lichtsignal,369
8,Affoltern,Strasse/Trottoir/Platz,327
9,Affoltern,VBZ/ÖV,71
